In [1]:
!pip install langchain-google-genai
!pip install pandas


In [2]:
import os
from dotenv import load_dotenv

# بارگذاری متغیرها از فایل .env
load_dotenv()

# اگر کلید با نام GEMINI_API_KEY ذخیره شده باشد، آن را در GOOGLE_API_KEY هم قرار می‌دهیم
if "GOOGLE_API_KEY" not in os.environ and "GEMINI_API_KEY" in os.environ:
    os.environ["GOOGLE_API_KEY"] = os.environ["GEMINI_API_KEY"]

In [3]:
import json

# Read the file "gods.json" and load it into the variable "gods"
with open("data/gods.json", "r", encoding="utf-8") as file:
    gods = json.load(file)

In [4]:
gods_info = "\n\n".join(
    [
        f"نام: {g['name']}\nتوضیحات: {g['description']}\nظاهر: {g['appearance']}\nویژگی‌ها: {g['features']}"
        for g in gods
    ]
)

system_message = f"""شما یک دستیار هوشمند استخراج اطلاعات از متن هستید.
تنها بر اساس اطلاعات زیر به سوالات پاسخ دهید:

{gods_info}

قوانین سخت‌گیرانه پاسخ‌دهی:
1. طول پاسخ: پاسخ شما باید اکیداً بین ۱ تا ۵ کلمه باشد. از نوشتن جملات کامل و افعال خودداری کنید.
2. رعایت تعداد: اگر در صورت سوال تعداد مشخصی خواسته شده است (مثلاً "دو حیوان" یا "دو ویژگی")، دقیقاً و فقط به همان تعداد از متن انتخاب کنید تا طول جواب کوتاه بماند.
3. مدیریت سوالات توصیفی (چگونه/چه نوع/ظاهر): کلماتی مثل «چگونه» یا «چه نوع» همیشه به معنای جواب طولانی نیستند. اگر پاسخِ این سوالات در متن به صورت یک عبارت کوتاه (مانند "از سر زئوس"، "پیرمرد با ریش بلند" یا "جنگجوی سالم یا جوان") وجود دارد، حتماً همان عبارت کوتاه را بنویسید.
4. قانون «نمی‌دانم»: تنها در دو حالت کلمه «نمی‌دانم» را چاپ کنید:
   الف) پاسخ سوال در متن بالا وجود نداشته باشد.
   ب) توصیف ظاهر یک خدا در متن به قدری طولانی باشد که استخراج یک عبارت ۱ تا ۵ کلمه‌ای از آن ممکن نباشد (فقط در این حالت تسلیم شوید).
5. پاسخ ها را فهرست وار ذکر کن
"""




In [5]:
examples = [
    {"question": "پادشاه خدایان کیست؟", "answer": "زئوس"},
    {"question": "مادر زئوس چه کسی است؟", "answer": "ریا"},
    {"question": "زئوس در دست چپ خود چه چیزی حمل می‌کند؟", "answer": "عصای سلطنتی"},
    {"question": "پوسوئیدون اغلب با چه نامی شناخته می‌شود؟", "answer": "نپتون"},
    {"question": "آرام‌ترین خدایان چه کسی است؟", "answer": "دمتر"},
    {"question": "دمتر به عنوان خدای چه کسانی در نظر گرفته شده است؟", "answer": "کشاورزان"},
    {"question": "مادر آرس چه کسی است؟", "answer": "هرا"},
    {"question": "هفایستوس بر روی چه حیوانی نشسته است؟", "answer": "الاغ"},
    {"question": "آگاه‌ترین خدای یونانی کیست؟", "answer": "آپولون"},
    {"question": "آپولون با چه چیزی در آسمان همراه است؟", "answer": "خورشید"},
    {"question": "آرتمیس بسیاری از اوقات با چه حیوانی همراه است؟", "answer": "گوزن"},
    {"question": "آتنا حامی کدام شهر است؟", "answer": "آتن"}
]

In [6]:
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate

example_prompt = ChatPromptTemplate.from_messages([
    ("human", "{question}"),
    ("ai", "{answer}")
])

few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_message),
    few_shot_prompt,
    ("human", "{question}")
])

In [7]:
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(
    model="gemini-flash-lite-latest",  # یا "gemini-1.5-flash" / "gemini-2.0-flash"
    temperature=0
)

In [8]:
from langchain_core.output_parsers import StrOutputParser

chain = prompt | model | StrOutputParser()

In [9]:
!pip install rich
!pip install markdown

In [10]:
import pandas as pd
import time
import markdown
from IPython.display import display, HTML, update_display
from langchain_core.output_parsers import StrOutputParser

chain = prompt | model | StrOutputParser()
questions = pd.read_csv("data/questions.csv")

def render(q_num, q_text, ans_md):
    body = markdown.markdown(ans_md, extensions=["fenced_code", "tables", "nl2br"])
    return f"""
    <div dir="rtl" style="text-align:right; font-family:Tahoma, Vazirmatn, sans-serif;
         line-height:1.9; border:1px solid #888; border-radius:10px;
         padding:12px 16px; margin:12px 0;">
      <div style="color:#0a84c6; font-weight:bold; margin-bottom:6px;">
        سوال {q_num}: {q_text}
      </div>
      <hr style="opacity:.3">
      <div>{body}</div>
    </div>
    """

answers = []
for i, row in questions.iterrows():
    q_text = row["question"]
    ans = ""
    display_id = f"ans_{i}"
    display(HTML(render(i, q_text, "...")), display_id=display_id)

    for chunk in chain.stream({"question": q_text}):
        ans += chunk
        update_display(HTML(render(i, q_text, ans)), display_id=display_id)

    answers.append(ans.strip())

    if i > 0 and i % 10 == 0:
        time.sleep(60)

submission = pd.DataFrame({"answer": answers})

In [11]:
import zipfile
import os

if not os.path.exists(os.path.join(os.getcwd(), '01_Project copy.ipynb')):
    %notebook -e Olympians.ipynb

def compress(file_names): 
    print("File Paths:")
    print(file_names)
    compression = zipfile.ZIP_DEFLATED
    with zipfile.ZipFile("result.zip", mode="w") as zf:
        for file_name in file_names:
            zf.write('./' + file_name, file_name, compress_type=compression)

configs = {
    "messages": len(prompt.messages),
    "temperature": model.temperature
}

with open('model_configs.json', 'w') as json_file:
    json.dump(configs, json_file, indent=4)

submission.to_csv('submission.csv', index=False)
pd.DataFrame(examples).to_csv('examples.csv', index=False)

file_names = ['Olympians.ipynb', 'submission.csv',
              'examples.csv','model_configs.json' ]
compress(file_names)

File Paths:
['Olympians.ipynb', 'submission.csv', 'examples.csv', 'model_configs.json']
